In [1]:
import pandas as pd
import json

In [2]:
dataset_path = r"F:\dev\infy\Week 1\dataset\open-data\data"

file_path = dataset_path + "/competitions.json"

with open(file_path, encoding="utf-8") as f:
    data = json.load(f)

df_comp = pd.DataFrame(data)


print("Shape:", df_comp.shape)
print("Columns:", df_comp.columns.tolist())
print()
print(df_comp.info())

Shape: (75, 12)
Columns: ['competition_id', 'season_id', 'country_name', 'competition_name', 'competition_gender', 'competition_youth', 'competition_international', 'season_name', 'match_updated', 'match_updated_360', 'match_available_360', 'match_available']

<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   competition_id             75 non-null     int64
 1   season_id                  75 non-null     int64
 2   country_name               75 non-null     str  
 3   competition_name           75 non-null     str  
 4   competition_gender         75 non-null     str  
 5   competition_youth          75 non-null     bool 
 6   competition_international  75 non-null     bool 
 7   season_name                75 non-null     str  
 8   match_updated              75 non-null     str  
 9   match_updated_360          57 non-null     str  

In [3]:
print("Non-empty Values:\n",df_comp.count())
print()
print(df_comp.head())

Non-empty Values:
 competition_id               75
season_id                    75
country_name                 75
competition_name             75
competition_gender           75
competition_youth            75
competition_international    75
season_name                  75
match_updated                75
match_updated_360            57
match_available_360          11
match_available              75
dtype: int64

   competition_id  season_id country_name        competition_name  \
0               9        281      Germany           1. Bundesliga   
1               9         27      Germany           1. Bundesliga   
2            1267        107       Africa  African Cup of Nations   
3              16          4       Europe        Champions League   
4              16          1       Europe        Champions League   

  competition_gender  competition_youth  competition_international  \
0               male              False                      False   
1               male        

In [1]:
import json
import pandas as pd
from pathlib import Path

DATASET_PATH = Path(r"F:\dev\infy\Week 1\dataset\open-data\data")

COMPETITION_ID = 11
SEASON_ID      = 90

with open(DATASET_PATH / "competitions.json", encoding="utf-8") as f:
    competitions = json.load(f)

df_comp = pd.DataFrame(competitions)
print("Competitions shape:", df_comp.shape)
print(df_comp[["competition_id", "season_id", "competition_name", "season_name"]].head(10).to_string(index=False))

with open(DATASET_PATH / "matches" / str(COMPETITION_ID) / f"{SEASON_ID}.json", encoding="utf-8") as f:
    matches = json.load(f)

match_ids = [m["match_id"] for m in matches]
print(f"\nLoaded {len(match_ids)} matches — competition {COMPETITION_ID}, season {SEASON_ID}")

all_events = []
for i, mid in enumerate(match_ids, 1):
    print(f"  Reading match [{i}/{len(match_ids)}] ID: {mid}", end="\r")
    with open(DATASET_PATH / "events" / f"{mid}.json", encoding="utf-8") as f:
        events = json.load(f)
    for e in events:
        etype = e.get("type", {}).get("name")
        all_events.append({
            "match_id":         mid,
            "player":           e.get("player", {}).get("name"),
            "team":             e.get("team", {}).get("name"),
            "type":             etype,
            "minute":           e.get("minute"),
            "period":           e.get("period"),
            "shot_outcome":     e.get("shot", {}).get("outcome", {}).get("name"),
            "pass_goal_assist": e.get("pass", {}).get("goal_assist"),
            "pass_outcome":     e.get("pass", {}).get("outcome", {}).get("name"),
            "duel_outcome":     e.get("duel", {}).get("outcome", {}).get("name"),
            "sub_off":          e.get("substitution", {}).get("replacement", {}).get("name") if etype == "Substitution" else None,
        })

print(f"\nTotal events: {len(all_events)}")
df = pd.DataFrame(all_events)

players = df[df["player"].notna()]["player"].unique()
records = []

for player in players:
    pdf = df[df["player"] == player]
    team = pdf["team"].iloc[0]
    match_list = pdf["match_id"].unique()

    goals   = len(pdf[(pdf["type"] == "Shot") & (pdf["shot_outcome"] == "Goal")])
    assists = int(pdf["pass_goal_assist"].fillna(False).sum())

    minutes_played = 0
    for mid in match_list:
        mdf = df[df["match_id"] == mid]
        player_events = mdf[mdf["player"] == player]
        if player_events.empty:
            continue
        sub_on = mdf[(mdf["type"] == "Substitution") & (mdf["sub_off"] == player)]
        start_min = int(sub_on["minute"].iloc[0]) if not sub_on.empty else 0
        sub_off = mdf[(mdf["type"] == "Substitution") & (mdf["player"] == player)]
        if not sub_off.empty:
            end_min = int(sub_off["minute"].iloc[0])
        else:
            end_min = 120 if mdf["period"].max() > 2 else 90
        minutes_played += max(0, end_min - start_min)

    passes_total    = len(pdf[pdf["type"] == "Pass"])
    passes_complete = len(pdf[(pdf["type"] == "Pass") & (pdf["pass_outcome"].isna())])
    tackles_total   = len(pdf[pdf["type"] == "Duel"])
    tackles_won     = len(pdf[(pdf["type"] == "Duel") & (pdf["duel_outcome"].isin(["Won", "Success"]))])

    records.append({
        "player":            player,
        "team":              team,
        "matches":           len(match_list),
        "minutes_played":    minutes_played,
        "goals":             goals,
        "assists":           assists,
        "shots":             len(pdf[pdf["type"] == "Shot"]),
        "passes_total":      passes_total,
        "passes_complete":   passes_complete,
        "pass_accuracy_pct": round(passes_complete / passes_total * 100, 1) if passes_total else 0.0,
        "tackles_total":     tackles_total,
        "tackles_won":       tackles_won,
        "dribbles":          len(pdf[pdf["type"] == "Dribble"]),
        "interceptions":     len(pdf[pdf["type"] == "Interception"]),
        "fouls_committed":   len(pdf[pdf["type"] == "Foul Committed"]),
    })

result = pd.DataFrame(records).sort_values("goals", ascending=False).reset_index(drop=True)

output_path = DATASET_PATH.parent / "statsbomb_player_stats.csv"
result.to_csv(output_path, index=False)

print("\nTop 20 players by goals:")
print(result.head(20).to_string(index=False))
print(f"\nSaved {len(result)} players → {output_path}")

Competitions shape: (75, 12)
 competition_id  season_id       competition_name season_name
              9        281          1. Bundesliga   2023/2024
              9         27          1. Bundesliga   2015/2016
           1267        107 African Cup of Nations        2023
             16          4       Champions League   2018/2019
             16          1       Champions League   2017/2018
             16          2       Champions League   2016/2017
             16         27       Champions League   2015/2016
             16         26       Champions League   2014/2015
             16         25       Champions League   2013/2014
             16         24       Champions League   2012/2013

Loaded 35 matches — competition 11, season 90
  Reading match [35/35] ID: 3773477
Total events: 139030

Top 20 players by goals:
                                          player             team  matches  minutes_played  goals  assists  shots  passes_total  passes_complete  pass_accuracy